**Part 2 · LangChain** · step 11 of 23  
[← 10 First agent](10_first_agent.ipynb) · [🏠 Roadmap](../README.md) · [12 Agent structured output →](12_agent_structured_output.ipynb)

# 11 · Agent Memory with a Checkpointer

In notebook 08 we kept history by hand. Agents do it for you:

- **Checkpointer** (`InMemorySaver`) — saves the agent's state after every step.
- **`thread_id`** — the key under which a conversation is saved. Same id = same conversation.

```python
config = {"configurable": {"thread_id": "abc"}}
agent.invoke({"messages": [...]}, config=config)   # state for "abc" is loaded + saved
```

`InMemorySaver` lives in RAM (lost when the kernel restarts). For production use a persistent
checkpointer such as SQLite or Postgres.

In [ ]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="gpt-4o",
    system_prompt="You are a helpful AI assistant",
    checkpointer=InMemorySaver(),  # <- this is what gives the agent memory
)

# uuid7 = time-ordered unique id; any unique string works as a thread_id
config = {"configurable": {"thread_id": str(uuid7())}}

In [ ]:
resp = agent.invoke({"messages": [HumanMessage("I am Ujjawal, I like coffee")]}, config=config)
resp["messages"][-1].content

In [ ]:
# Same thread_id -> the agent remembers the previous turn
resp2 = agent.invoke({"messages": [HumanMessage("Tell me about myself")]}, config=config)
resp2["messages"][-1].content

In [ ]:
# A NEW thread_id is a brand new conversation -> no memory of Ujjawal
other_thread = {"configurable": {"thread_id": str(uuid7())}}
resp3 = agent.invoke({"messages": [HumanMessage("Tell me about myself")]}, config=other_thread)
resp3["messages"][-1].content

## Revision notes
- No checkpointer → every `invoke` starts from scratch.
- Checkpointer + `thread_id` → you send **only the new message**; old ones are loaded automatically.
- One checkpointer can hold many threads (one per user / chat session).

---
✅ **Done with 11 Agent memory.** Next: [12 Agent structured output →](12_agent_structured_output.ipynb)  
[🏠 Roadmap](../README.md)